# Checking the quality of LLM Answers for bulk input queries

# STAGE 1 : GENERATING OUTPUT

In [6]:
import google.generativeai as genai
import pandas as pd

/var/folders/f5/j995pwn90q3c6pyc8ts6mh_00000gn/T/ipykernel_5148/1754626231.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [ ]:
genai.configure(api_key = "YOUR_API_KEY_HERE")
aimodel   = "gemini-3.5-flash-lite"

In [12]:
 # FILE CONFIGURATION

INPUT_CSV = "ai_teacher_20_questions.csv"
OUTPUT_CSV = "ai_teacher_results.csv"

df = pd.read_csv(INPUT_CSV)
#print(df)
#print(f"Number of questions: {len(df)}")

In [7]:
# AI TEACHER INSTRUCTIONS

TEACHER_INSTRUCTIONS = """

You are an expert AI Teacher. Your job is to teach students by explaining concepts rather than just providing direct answers. 

Follow these strict rules:
1. Explain the concept in simple, layman's terms.
2. Limit your entire response to exactly 3 lines of text.
3. Formatting: Provide only the simple explanation. Do not include introductory text, conversational filler, or headers.
"""

In [13]:
# CONVERSATION MEMORY
conversation = [
    {
    "role"    : "user",
    "parts" : TEACHER_INSTRUCTIONS
    }
]

In [22]:
#GENERATE ANSWER FOR ALL QUESTIONS

def generate_answer(question):
    conversation.append({"role": "user", "parts": [f"Student Question: {question}"]})

    model = genai.GenerativeModel(aimodel)
    response = model.generate_content(
        conversation,
        generation_config={"temperature": 0},
    )

    answer = response.text.strip()
    conversation.append({"role": "model", "parts": [answer]})
    return answer

In [23]:
generated_answers = []

for index, row in df.iterrows():

    question = row["User Query"]

    print(f"Processing {index + 1}/{len(df)}")
    print(f"Question: {question}")

    try:

        answer = generate_answer(question)

    except Exception as e:

        answer = f"ERROR: {str(e)}"

    generated_answers.append(answer)

Processing 1/20
Question: What is AI?
Processing 2/20
Question: How does AI differ from traditional programming?
Processing 3/20
Question: What is Machine Learning?
Processing 4/20
Question: What is Deep Learning?
Processing 5/20
Question: What is the difference between AI, ML, and Deep Learning?
Processing 6/20
Question: What is training data?
Processing 7/20
Question: What is a Machine Learning model?
Processing 8/20
Question: What is supervised learning?
Processing 9/20
Question: What is unsupervised learning?
Processing 10/20
Question: What is reinforcement learning?
Processing 11/20
Question: What is an LLM?
Processing 12/20
Question: What is Generative AI?
Processing 13/20
Question: What is a prompt?
Processing 14/20
Question: What is Prompt Engineering?
Processing 15/20
Question: What is RAG?
Processing 16/20
Question: Why is RAG useful?
Processing 17/20
Question: What is a vector database?
Processing 18/20
Question: What are embeddings?
Processing 19/20
Question: What is an AI 

In [25]:
# APPEND GENERATED ANSWER TO DATASET
df["GeneratedAnswer"] = generated_answers
#print(generated_answers)

In [26]:
# SAVE RESULT
df = df[ [ "User Query", "Teacher Response", "GeneratedAnswer" ] ]

df.to_csv( OUTPUT_CSV, index=False, encoding="utf-8-sig")

In [27]:
# DISPLAY RESULTS

print("\n" + "=" * 70)
print("AI TEACHER TEST COMPLETED")
print("=" * 70)

print(df.to_string(index=False))

print("\nResult saved to:")
print(OUTPUT_CSV)


AI TEACHER TEST COMPLETED
                                               User Query                                                                                                                                                                             Teacher Response                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

# STAGE 2 : EVALUATE ANSWER QUALITY

In [1]:
#pip install deepeval

In [14]:
import pandas as pd

from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric, GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.evaluate import AsyncConfig

from deepeval.models import GeminiModel
from deepeval.test_case import LLMTestCaseParams

/var/folders/f5/j995pwn90q3c6pyc8ts6mh_00000gn/T/ipykernel_5148/555647321.py:9: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams


In [15]:
#Load data
df2 = pd.read_csv("ai_teacher_results.csv")
#df2

In [16]:
# Create Test Cases

test_cases = [ LLMTestCase(input = row["User Query"], actual_output = row["GeneratedAnswer"], expected_output = row["Teacher Response"]) 
             for _, row in df2.iterrows() ]

In [ ]:
gemini = GeminiModel(
    model="gemini-3.5-flash-lite",   # check the name, see point 2
    api_key=" ",   # or set GOOGLE_API_KEY in the environment
    temperature=0,
)

In [18]:
#Defining Metrics

#Correctness
correctness = GEval(
    name               = "Answer Correctness",
    model              = gemini,
    criteria           = """ 
        Evaluate whether the generated answer is factually correct compared with the expected teacher answer.
        Give a high score when the important meaning and facts are correct.

        Penalize:
        - incorrect facts
        - contradictory information
        - missing important information

        Minor wording differences should not reduce the score.
        """,
    evaluation_params = [LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
    async_mode = False
)

# Relevancy
relevancy = AnswerRelevancyMetric(model=gemini, include_reason=False, async_mode=False)


#Helpfulness
helpfulness = GEval(
    name               = "Helpfulness",
    model              = gemini,
    criteria           = """ 
        Evaluate how helpful the generated answer is for a beginner student.

        A helpful answer should:

        1. Directly answer the student's question.
        2. Be easy to understand.
        3. Provide enough explanation.
        4. Avoid unnecessary information.
        5. Help the student understand the concept.
        """,
    evaluation_params = [LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    async_mode = False
)

In [25]:
# Run DeepEval

result = evaluate(test_cases = test_cases, 
                  metrics = [correctness, relevancy, helpfulness],                   
                  async_config=AsyncConfig(run_async=True, max_concurrent=1, throttle_value=30)
                 )

✨ You're running DeepEval's latest Answer Correctness [GEval] Metric! (using gemini-3.5-flash-lite (Gemini), 
strict=False, async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gemini-3.5-flash-lite (Gemini), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Helpfulness [GEval] Metric! (using gemini-3.5-flash-lite (Gemini), 
strict=False, async_mode=True)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_2 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_3 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_4 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_5 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_6 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_7 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_8 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_9 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_10 (Passed 3 metrics)                                

⚠ WARNING: No hyperparameters logged.
» ]8;id=503719;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 600.1s | token cost: 0.026302400000000007 USD)
» Test Results (20 total tests):
   » Pass Rate: 75.0% | Passed: 15 | Failed: 5

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [27]:
for i, test_result in enumerate(result.test_results):

    metrics_data = test_result.metrics_data

    df2.loc[i, "AnswerCorrectness"] = metrics_data[0].score
    df2.loc[i, "AnswerRelevancy"]   = metrics_data[1].score
    df2.loc[i, "Helpfulness"]       = metrics_data[2].score

In [28]:
# Calculate overall answer quality
df2["AnswerQuality"] = (df2["AnswerCorrectness"] + df2["AnswerRelevancy"] + df2["Helpfulness"])/3

In [31]:
# Round up the scores
metric_columns = ["AnswerCorrectness", "AnswerRelevancy", "Helpfulness", "AnswerQuality"]

df2[metric_columns] = df2[metric_columns].round(3)

#Average the scores
average_correctness = df2["AnswerCorrectness"].mean()
average_relevancy   = df2["AnswerRelevancy"].mean()
average_helpfulness = df2["Helpfulness"].mean()
average_quality     = df2["AnswerQuality"].mean()

In [33]:
#Save results
df2.to_csv("ai_teacher_evaluation.csv", index=False, encoding="utf-8-sig")

In [36]:
# Result for each question
print("\n")
print("=" * 120)
print("                         AI TEACHER ANSWER QUALITY")
print("=" * 120)

display_df = df2[ ["User Query", "AnswerCorrectness", "AnswerRelevancy", "Helpfulness", "AnswerQuality"] ]
df2



                         AI TEACHER ANSWER QUALITY


,User Query,Teacher Response,GeneratedAnswer,AnswerCorrectness,AnswerRelevancy,Helpfulness,AnswerQuality
0,What is AI?,AI (Artificial Intelligence) is the ability of...,Artificial intelligence is like teaching a com...,0.9,1.000,1.0,0.967
1,How does AI differ from traditional programming?,Traditional programs follow fixed rules writte...,Traditional programming is like following a st...,1.0,1.000,1.0,1.000
2,What is Machine Learning?,Machine Learning is a branch of AI where compu...,Machine learning\n is a method where computers...,1.0,1.000,1.0,1.000
3,What is Deep Learning?,Deep Learning is a type of Machine Learning th...,Deep learning is an advanced type of machine l...,1.0,1.000,1.0,1.000
4,"What is the difference between AI, ML, and Dee...",AI is the broad field of making machines intel...,Think of these concepts like Russian nesting d...,1.0,1.000,1.0,1.000
5,What is training data?,Training data is the data used to teach a Mach...,Training data is the collection of examples yo...,0.8,1.000,1.0,0.933
6,What is a Machine Learning model?,A Machine Learning model is a mathematical sys...,A machine learning model is the brain-like mat...,1.0,1.000,1.0,1.000
7,What is supervised learning?,Supervised learning trains a model using examp...,Supervised learning is like teaching a student...,1.0,1.000,1.0,1.000
8,What is unsupervised learning?,Unsupervised learning works with data that doe...,Unsupervised learning is like giving a compute...,1.0,1.000,1.0,1.000
9,What is reinforcement learning?,Reinforcement learning teaches an agent throug...,Reinforcement learning is like training a pet ...,1.0,1.000,1.0,1.000


In [37]:
average_result = pd.DataFrame(columns=["average_correctness","average_relevancy","average_helpfulness","average_quality"],
                            data=[[average_correctness,average_relevancy,average_helpfulness,average_quality]])

average_result

,average_correctness,average_relevancy,average_helpfulness,average_quality
0,0.73,0.79645,0.83,0.7855
